# Mega analysis: real-profit decisions, model selection, label delay, robustness, 60-month replication
Everything here uses **real Lending Club loans** (no simulation). Run All; expect roughly 15-40 minutes. Each section is wrapped so that one failure does not stop the rest; the end of the notebook lists any section that failed and zips all results to `Desktop/Q2/outputs_mega.zip` (send me that zip).

**Needs** (all in `Desktop/Q2`): `extra_data/lending_club_36m_wide.csv`, `extra_data/lending_club_60m_wide.csv` and, for the profit sections, the raw Kaggle file in `archive/` (read once to pull `total_pymnt`, cached as `extra_data/payments.csv`).

| # | section | question |
|---|---|---|
| 1 | model zoo | 3 models x 4 feature sets; selected on a **validation year (2011)**, never on test years |
| 2 | conformal methods | static, class-conditional, bin-conditional, rolling recalibration and ACI (step sizes scaled for monthly batch feedback) under 0/6/12-month label delay |
| 3 | decisions on **realised profit** | approve-all, tuned cutoff, conformal tiers (review rules and review costs swept), with month-block bootstrap intervals; replaces the invented cost model |
| 4 | robustness | other calibration years, random-split controls (is the shortfall temporal?), alpha sensitivity |
| 5 | 60-month replication | train 2010-11, calibrate 2012, test 2013 by quarter |

**Assumptions to state in the paper.** Profit = `total_pymnt - funded_amnt` per loan: undiscounted, before platform fees, observed only for loans LendingClub accepted (so this is a *second-stage screen on accepted loans*, not an acceptance model). Tuned policies use the 2012 vintage, whose outcomes would not all be known on 1 Jan 2013 (36-month loans), so tuned policies are optimistic by construction. The manual-review cost is an **assumption** and is swept, not fixed.

In [ ]:
import os, sys, subprocess, importlib, warnings, json, gzip, zipfile, hashlib, time, traceback, contextlib, shutil
from pathlib import Path
def ensure(pkg, mod=None):
    try: importlib.import_module(mod or pkg)
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
ensure('lightgbm'); ensure('statsmodels'); ensure('scikit-learn', 'sklearn')
import numpy as np, pandas as pd, lightgbm as lgb, matplotlib
matplotlib.use(os.environ.get('MPL_BACKEND', matplotlib.get_backend()))
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss
from sklearn.linear_model import LogisticRegression
from statsmodels.stats.proportion import proportion_confint
warnings.filterwarnings('ignore'); plt.style.use('seaborn-v0_8-whitegrid')

def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2(); assert Q2 is not None, "No 'Q2' folder on the Desktop; set Q2 = Path(r'...') here."
OUT = Q2/'outputs_mega'; TAB, FIG = OUT/'tables', OUT/'figures'; TAB.mkdir(parents=True, exist_ok=True); FIG.mkdir(exist_ok=True)
SEED, ALPHA = 42, 0.10
MAX_PER_MONTH = int(os.environ.get('MAX_PER_MONTH', 2500))      # used only by the row-by-row online methods (section 2); everything else uses ALL loans
FAILED = []; T0 = time.time()

@contextlib.contextmanager
def section(name):
    t = time.time(); print(f"\n{'=' * 10} {name} {'=' * 10}")
    try: yield
    except Exception:
        FAILED.append(name); print(f"!!! SECTION FAILED: {name}"); traceback.print_exc()
    finally: print(f"[{name}: {time.time() - t:.0f}s | total {(time.time() - T0) / 60:.1f} min]")
def save(df, name): df.round(5).to_csv(TAB/f'{name}.csv', index=False)
def savefig(name):
    for ext in ('png', 'pdf'): plt.savefig(FIG/f'{name}.{ext}', dpi=180, bbox_inches='tight')

# ---------- conformal helpers (same definitions as the main notebook) ----------
def conf_q(scores, alpha):
    s = np.sort(np.asarray(scores)); n = len(s); k = int(np.ceil((n + 1) * (1 - alpha)))
    return np.inf if k > n else s[k - 1]
def true_scores(p, y): return np.where(y == 1, 1 - p, p)
def conformal_sets(p, y, ca, alpha=ALPHA, nbin=5):
    sc = true_scores(p, y); q = conf_q(sc[ca], alpha); q0, q1 = conf_q(sc[ca & (y == 0)], alpha), conf_q(sc[ca & (y == 1)], alpha)
    edges = np.quantile(p[ca], np.linspace(0, 1, nbin + 1)[1:-1]); b = np.digitize(p, edges)
    qb = np.array([conf_q(sc[ca & (b == k)], alpha) for k in range(nbin)])[b]
    return {'static': np.column_stack([p <= q, (1 - p) <= q]), 'class-conditional': np.column_stack([p <= q0, (1 - p) <= q1]),
            'bin-conditional': np.column_stack([p <= qb, (1 - p) <= qb])}
def cov_stats(S, y):
    hit = S[np.arange(len(y)), y]; ap, rj = S[:, 0] & ~S[:, 1], S[:, 1] & ~S[:, 0]
    return dict(cov=hit.mean(), cov_default=hit[y == 1].mean(), cov_nondefault=hit[y == 0].mean(), size=S.sum(1).mean(),
                share_approve_only=ap.mean(), share_reject_only=rj.mean(), share_review=1 - ap.mean() - rj.mean(), share_empty=(~S.any(1)).mean())
def wilson(k, n): return proportion_confint(int(k), int(n), method='wilson')
print("Q2:", Q2, "| outputs:", OUT)

In [ ]:
# ---------- 0. data: wide files, payments (profit), per-month sample indices ----------
with section('0. load data'):
    W36 = pd.read_csv(Q2/'extra_data'/'lending_club_36m_wide.csv', dtype={'id': str})
    P60 = Q2/'extra_data'/'lending_club_60m_wide.csv'; W60 = pd.read_csv(P60, dtype={'id': str}) if P60.exists() else None
    man = Q2/'extra_data'/'raw_manifest.json'
    if man.exists():
        mj = json.loads(man.read_text()); h = hashlib.sha256((Q2/'extra_data'/'lending_club_36m_wide.csv').read_bytes()).hexdigest()
        print("36m wide file matches manifest hash:", h == mj['outputs'][0]['sha256'], "| raw sha256:", mj['raw_sha256'][:16], "...")

    def find_raw():
        arc = Q2/'archive'; files = [p for p in arc.rglob('*') if p.is_file()] if arc.is_dir() else []
        acc = [p for p in files if 'accepted' in p.name.lower() and p.name.lower().endswith(('.csv', '.csv.gz'))]
        if acc: return max(acc, key=lambda p: p.stat().st_size)
        for z in list(Q2.glob('*.zip')) + [p for p in files if p.suffix.lower() == '.zip']:
            with zipfile.ZipFile(z) as zf:
                hit = [n for n in zf.namelist() if 'accepted' in n.lower() and n.lower().endswith(('.csv', '.csv.gz'))]
                if hit: return (z, hit[0])
        return None
    def open_raw(raw):
        if isinstance(raw, tuple):
            f = zipfile.ZipFile(raw[0]).open(raw[1]); return gzip.open(f) if raw[1].lower().endswith('.gz') else f
        return raw
    PAYF = Q2/'extra_data'/'payments.csv'
    if PAYF.exists(): pay = pd.read_csv(PAYF, dtype={'id': str}); print("Using cached", PAYF)
    else:
        raw = find_raw(); pay = None
        if raw is None: print("!! raw Kaggle file not found in Q2/archive: profit sections will be skipped")
        else:
            ids = set(W36['id']) | (set(W60['id']) if W60 is not None else set()); parts = []
            print("Reading payments from raw file (a few minutes, once)...")
            for ch in pd.read_csv(open_raw(raw), usecols=['id', 'total_pymnt', 'funded_amnt'], dtype={'id': str}, chunksize=250_000, low_memory=False):
                parts.append(ch[ch['id'].isin(ids)])
            pay = pd.concat(parts).drop_duplicates('id'); pay.to_csv(PAYF, index=False); print("Saved", PAYF, len(pay))
    HAS_PAY = False
    if pay is not None:
        pay = pay.drop_duplicates('id')[['id', 'total_pymnt']]
        W36 = W36.merge(pay, on='id', how='left'); W36['profit'] = W36['total_pymnt'] - W36['funded_amnt']
        if W60 is not None: W60 = W60.merge(pay, on='id', how='left'); W60['profit'] = W60['total_pymnt'] - W60['funded_amnt']
        HAS_PAY = W36['total_pymnt'].notna().mean() > 0.99
        print(f"Payments matched for {W36['total_pymnt'].notna().mean():.2%} of 36-month loans | mean profit per loan ${W36['profit'].mean():,.0f} | "
              f"ROI on funded {W36['profit'].sum() / W36['funded_amnt'].sum():.2%}")
        print("Mean profit per $ funded by outcome:", W36.groupby('default').apply(lambda d: d['profit'].sum() / d['funded_amnt'].sum()).round(3).to_dict())

    full = W36[W36['year'] <= 2015].reset_index(drop=True)
    for d in (full,) + ((W60,) if W60 is not None else ()):
        d['grade_ord'] = d['grade'].map({g: i for i, g in enumerate('ABCDEFG')})
        d['subgrade_ord'] = d['sub_grade'].map({f"{g}{k}": 5 * i + k - 1 for i, g in enumerate('ABCDEFG') for k in range(1, 6)})
    y = full['default'].values; yr = full['year'].values; tmon = ((full['year'] - 2007) * 12 + full['month']).values
    sh = full.sample(frac=1.0, random_state=SEED)
    keep = sh[sh.groupby(['year', 'month']).cumcount() < MAX_PER_MONTH] if MAX_PER_MONTH else sh
    samp_idx = keep.sort_values('issue_d', kind='stable').index.values          # per-month sample, random within-month order
    print(f"All loans: {len(full):,} | online-method sample: {len(samp_idx):,} | default rate {y.mean():.2%}")

In [ ]:
# ---------- 1. model zoo: 3 models x 4 feature sets, selected on validation year 2011 ----------
BASE_NUM = ['loan_amnt', 'annual_inc', 'dti', 'fico_score', 'revol_util', 'delinq_2yrs', 'emp_length_yrs']
BASE_CAT = ['home_ownership', 'verification_status', 'purpose']
BUREAU = ['open_acc', 'pub_rec', 'total_acc', 'inq_last_6mths', 'mths_since_last_delinq', 'pub_rec_bankruptcies', 'revol_bal', 'credit_hist_yrs']
FS = {'F0 with int_rate': (BASE_NUM + ['int_rate'], BASE_CAT), 'F1 no lender score': (BASE_NUM, BASE_CAT),
      'F3 applicant-only rich': (BASE_NUM + BUREAU, BASE_CAT + ['addr_state']),
      'F4 everything': (BASE_NUM + BUREAU + ['int_rate', 'grade_ord', 'subgrade_ord', 'installment'], BASE_CAT + ['addr_state'])}
MODELS = ['LGBM default', 'LGBM regularised', 'Logistic (L2)']
def design(d, nums, cats): return pd.get_dummies(d[nums + cats], columns=cats, drop_first=True)

def fit_predict(kind, X, y_, mtr):
    if kind == 'LGBM default':
        m = lgb.LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=SEED, verbose=-1).fit(X[mtr], y_[mtr]); return m.predict_proba(X)[:, 1]
    if kind == 'LGBM regularised':      # fixed a priori (not tuned on any evaluation year)
        m = lgb.LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15, max_depth=4, min_child_samples=200, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.7, reg_lambda=10.0, random_state=SEED, verbose=-1).fit(X[mtr], y_[mtr]); return m.predict_proba(X)[:, 1]
    Xf = X.astype('float32'); miss = [c for c in Xf.columns if Xf.loc[mtr, c].isna().any()]
    ind = Xf[miss].isna().astype('float32').add_suffix('_na'); Xf = pd.concat([Xf.fillna(Xf.loc[mtr].median()), ind], axis=1)
    mu, sd = Xf.loc[mtr].mean(), Xf.loc[mtr].std().replace(0, 1); Z = ((Xf - mu) / sd).values
    return LogisticRegression(C=0.1, max_iter=3000).fit(Z[mtr], y_[mtr]).predict_proba(Z)[:, 1]

with section('1. model zoo'):
    P = {}; zoo = []; Xc = {}
    for fs, (nums, cats) in FS.items():
        Xc[fs] = design(full, nums, cats)
        for kind in MODELS:
            p_val = fit_predict(kind, Xc[fs], y, yr <= 2010)                        # trained on <= 2010, scored on 2011 for selection
            v = yr == 2011; val_ll = log_loss(y[v], np.clip(p_val[v], 1e-6, 1 - 1e-6))
            P[(fs, kind)] = fit_predict(kind, Xc[fs], y, yr <= 2011)                 # final model: trained on <= 2011
            ca = yr == 2012; S = conformal_sets(P[(fs, kind)], y, ca)
            row = dict(feature_set=fs, model=kind, n_features=Xc[fs].shape[1], val_logloss_2011=val_ll)
            for yy in (2012, 2013, 2014, 2015):
                m = yr == yy; pp = P[(fs, kind)][m]
                row.update({f'AUC_{yy}': roc_auc_score(y[m], pp), f'calib_gap_pp_{yy}': 100 * (y[m].mean() - pp.mean())})
                if yy >= 2013:
                    cs = cov_stats(S['static'][m], y[m]); cm = cov_stats(S['class-conditional'][m], y[m])
                    row.update({f'static_cov_{yy}': cs['cov'], f'static_size_{yy}': cs['size'], f'classcond_cov_default_{yy}': cm['cov_default'], f'classcond_size_{yy}': cm['size']})
            zoo.append(row); print(f"{fs:26s} {kind:18s} val logloss {val_ll:.4f}  AUC2015 {row['AUC_2015']:.3f}  static cov 2013-15: "
                                   f"{row['static_cov_2013']:.3f}/{row['static_cov_2014']:.3f}/{row['static_cov_2015']:.3f}")
    ZOO = pd.DataFrame(zoo); save(ZOO, 'M1_model_zoo')
    best = ZOO.loc[ZOO['val_logloss_2011'].idxmin()]
    PIPES = {'paper: F0 + LGBM default': ('F0 with int_rate', 'LGBM default')}
    if (best.feature_set, best.model) != ('F0 with int_rate', 'LGBM default'): PIPES[f'selected: {best.feature_set} + {best.model}'] = (best.feature_set, best.model)
    print("\nSelected by lowest 2011 validation log loss:", best.feature_set, '+', best.model, "| pipelines carried forward:", list(PIPES))

In [ ]:
# ---------- 2. conformal methods under label delay (row-by-row online methods on the per-month sample) ----------
def aci(p, y_, month, cal_scores, alpha, gamma, delay=0, clip=(0.01, 0.99)):
    s = np.sort(cal_scores); n = len(s); T = len(y_); a = alpha; err = np.zeros(T); size = np.zeros(T); ptr = 0
    for t in range(T):
        while ptr < t and month[ptr] + delay <= month[t]:
            a = np.clip(a + gamma * (alpha - err[ptr]), *clip); ptr += 1
        k = int(np.ceil((n + 1) * (1 - a))); q = -np.inf if k <= 0 else (np.inf if k > n else s[k - 1])
        in0, in1 = p[t] <= q, (1 - p[t]) <= q; err[t] = 1 - (in1 if y_[t] == 1 else in0); size[t] = int(in0) + int(in1)
    return 1 - err, size

def rolling_recal(p, y_, month, is_test, pool_ok, alpha, window, delay, cal_scores, min_pool=300):
    sc = true_scores(p, y_); T = len(y_); cov = np.full(T, np.nan); size = np.full(T, np.nan); fallback = np.zeros(T, bool)
    for m in np.unique(month[is_test]):
        idx = np.where(is_test & (month == m))[0]
        pool = sc[pool_ok & (month >= m - delay - window + 1) & (month <= m - delay)]            # labels already revealed, issued in the last `window` months
        if len(pool) < min_pool: pool = cal_scores; fallback[idx] = True
        q = conf_q(pool, alpha); S = np.column_stack([p[idx] <= q, (1 - p[idx]) <= q])
        cov[idx] = S[np.arange(len(idx)), y_[idx]]; size[idx] = S.sum(1)
    return cov[is_test], size[is_test], fallback[is_test]

DELAY_RES = {}
with section('2. conformal methods under label delay'):
    ys, yrs_, ms = y[samp_idx], yr[samp_idx], tmon[samp_idx]
    ca_s, te_s = yrs_ == 2012, yrs_ >= 2013
    NPM = pd.Series(ms[te_s]).value_counts().mean(); r1 = lambda x: float(f"{x:.1g}")
    G_IMM, G_DEL = 0.015, [r1(0.1 / NPM), r1(0.25 / NPM)]            # delayed grids set from the BATCH gain gamma x loans/month in {0.1, 0.25}
    print(f"loans/month in sample ~ {NPM:.0f}; delayed ACI step sizes {G_DEL} (batch gain 0.1 / 0.25); immediate-feedback gamma {G_IMM}")
    for pname, key in PIPES.items():
        p_s = P[key][samp_idx]; sc_s = true_scores(p_s, ys); cal_sc = sc_s[ca_s]
        S_all = conformal_sets(p_s, ys, ca_s); rows = []
        def add(method, delay, cov_, size_, S_=None):
            for lab, m in (('2013', yrs_[te_s] == 2013), ('2014', yrs_[te_s] == 2014), ('2015', yrs_[te_s] == 2015), ('2013-15', np.ones(te_s.sum(), bool))):
                yy = ys[te_s][m]; c = cov_[m]
                rows.append(dict(pipeline=pname, method=method, delay_months=delay, period=lab, cov=c.mean(), cov_default=c[yy == 1].mean(), size=size_[m].mean()))
        for lab in ('static', 'class-conditional', 'bin-conditional'):
            S = S_all[lab][te_s]; hit = S[np.arange(S.shape[0]), ys[te_s]].astype(float); add(f'{lab} (2012 calibration)', '-', hit, S.sum(1).astype(float))
        for d in (6, 12):
            c_, s_, fb_ = rolling_recal(p_s, ys, ms, te_s, yrs_ >= 2012, ALPHA, 12, d, cal_sc)
            add('rolling recalibration (W=12 mo)', d, c_, s_)
        c_, s_ = aci(p_s[te_s], ys[te_s], ms[te_s], cal_sc, ALPHA, G_IMM, 0); add(f'ACI gamma={G_IMM} (immediate feedback)', 0, c_, s_)
        for g in G_DEL:
            for d in (6, 12):
                c_, s_ = aci(p_s[te_s], ys[te_s], ms[te_s], cal_sc, ALPHA, g, d); add(f'ACI gamma={g:g}', d, c_, s_)
        DELAY_RES[pname] = pd.DataFrame(rows); print(f"\n{pname}"); print(DELAY_RES[pname][DELAY_RES[pname].period.isin(['2015', '2013-15'])].round(3).to_string(index=False))
    save(pd.concat(DELAY_RES.values()), 'M2_methods_under_delay')

In [ ]:
# ---------- 3. decisions on REALISED profit (all loans; the profit of each loan is real) ----------
def tiers(S): ap = S[:, 0] & ~S[:, 1]; rj = S[:, 1] & ~S[:, 0]; return ap, ~(ap | rj), rj          # approve-only, review (incl. empty), reject-only
REVIEW_COSTS = (0.0, 0.005, 0.01, 0.02)                                                           # expert-review cost as a fraction of the funded amount: ASSUMPTION, swept
DEC = {}
with section('3. decisions on realised profit'):
    assert HAS_PAY, "payments not available (raw file missing?) -> skipping"
    pi, fund, dflt = full['profit'].values, full['funded_amnt'].values.astype(float), y
    ca, te = yr == 2012, yr >= 2013; rng = np.random.default_rng(SEED)
    blk = pd.factorize(yr[te] * 100 + full['month'].values[te])[0]; nblk = blk.max() + 1
    def contrib(ap, rv, rule, c=0.0):
        if rule == 'approve-only (review->reject)': return np.where(ap, pi, 0.0), ap
        if rule == 'review->approve': return np.where(ap | rv, pi, 0.0), ap | rv
        take = ap | (rv & (dflt == 0))                                                              # expert resolves the label perfectly, at cost c x funded
        return np.where(ap, pi, 0.0) + np.where(rv, np.where(dflt == 0, pi, 0.0) - c * fund, 0.0), take
    all_rows = []
    for pname, key in PIPES.items():
        p = P[key]; S = conformal_sets(p, y, ca)
        # cutoff tuned on 2012 profit
        grid = np.quantile(p[ca], np.linspace(0.05, 1.0, 96)); c_best = grid[int(np.argmax([pi[ca & (p <= g)].sum() for g in grid]))]
        # alpha tuned on 2012 profit (static conformal, review treated as reject)
        sc = true_scores(p, y); best_a, best_v = None, -np.inf
        for a in np.arange(0.01, 0.61, 0.01):
            q = conf_q(sc[ca], a); Sa = np.column_stack([p <= q, (1 - p) <= q]); ap_, rv_, _ = tiers(Sa[ca]); v = pi[ca][ap_].sum()
            if v > best_v: best_a, best_v = a, v
        qa = conf_q(sc[ca], best_a); S_tuned = np.column_stack([p <= qa, (1 - p) <= qa])
        pols = [('Approve all', np.ones(len(y), bool), np.zeros(len(y), bool), [None]),
                (f'p_hat cutoff <= {c_best:.3f} (tuned on 2012 profit)', p <= c_best, np.zeros(len(y), bool), [None]),
                (f'static conformal, alpha={best_a:.2f} (tuned on 2012 profit)', *tiers(S_tuned)[:2], ['approve-only (review->reject)']),
                ('Oracle: approve iff profitable (hindsight bound)', pi > 0, np.zeros(len(y), bool), [None])]
        for lab in ('static', 'class-conditional', 'bin-conditional'):
            ap, rv, _ = tiers(S[lab]); pols.append((f'{lab} conformal, alpha=0.10', ap, rv, ['approve-only (review->reject)', 'review->approve'] + [f'expert, cost {c:.1%}' for c in REVIEW_COSTS]))
        base = np.where(np.ones(len(y), bool), pi, 0.0)
        for pol, ap, rv, rules in pols:
            for rule in rules:
                if rule is None: cb, took = np.where(ap, pi, 0.0), ap; rname = 'single tier'
                elif rule.startswith('expert'): cb, took = contrib(ap, rv, 'expert', float(rule.split('cost ')[1].rstrip('%')) / 100); rname = rule
                else: cb, took = contrib(ap, rv, rule); rname = rule
                for lab_, m in (('2013', yr == 2013), ('2014', yr == 2014), ('2015', yr == 2015), ('2013-15', te)):
                    mm = m & te if lab_ != '2013-15' else te; n = mm.sum(); tk = took[mm]
                    row = dict(pipeline=pname, policy=pol, rule=rname, period=lab_, N=int(n), profit_per_applicant=cb[mm].sum() / n,
                               gain_vs_approve_all=(cb[mm].sum() - base[mm].sum()) / n, share_approved=tk.mean(), share_review=rv[mm].mean(),
                               roi_on_approved=pi[mm][tk].sum() / max(fund[mm][tk].sum(), 1), default_rate_approved=dflt[mm][tk].mean() if tk.any() else np.nan)
                    if lab_ == '2013-15':
                        a_ = np.bincount(blk, cb[te], nblk); A_ = np.bincount(blk, base[te], nblk); nb_ = np.bincount(blk, minlength=nblk)
                        ix = rng.integers(0, nblk, (1000, nblk)); d_ = (a_[ix].sum(1) - A_[ix].sum(1)) / nb_[ix].sum(1)
                        row['gain_ci_lo'], row['gain_ci_hi'] = np.percentile(d_, [2.5, 97.5])
                    all_rows.append(row)
    DEC = pd.DataFrame(all_rows); save(DEC, 'M3_decisions_realised_profit')
    pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
    show = DEC[DEC.period == '2013-15'][['pipeline', 'policy', 'rule', 'share_approved', 'share_review', 'profit_per_applicant', 'gain_vs_approve_all', 'gain_ci_lo', 'gain_ci_hi', 'roi_on_approved', 'default_rate_approved']]
    print(show.round(4).to_string(index=False))

In [ ]:
# ---------- 4. robustness: other calibration years, random-split controls, alpha sensitivity (all loans) ----------
with section('4a. rolling origin: other calibration years'):
    rows = []
    for fs in ('F0 with int_rate', 'F1 no lender score'):
        for Cy in (2011, 2012, 2013):
            p = fit_predict('LGBM default', Xc[fs], y, yr <= Cy - 1); ca_ = yr == Cy; S = conformal_sets(p, y, ca_)
            for yy in range(Cy + 1, 2016):
                m = yr == yy; cs = cov_stats(S['static'][m], y[m]); cm = cov_stats(S['class-conditional'][m], y[m]); lo, hi = wilson(cs['cov'] * m.sum(), m.sum())
                rows.append(dict(feature_set=fs, calib_year=Cy, test_year=yy, lead=yy - Cy, N=int(m.sum()), AUC=roc_auc_score(y[m], p[m]), calib_gap_pp=100 * (y[m].mean() - p[m].mean()),
                                 static_cov=cs['cov'], ci_lo=lo, ci_hi=hi, static_cov_default=cs['cov_default'], classcond_cov_default=cm['cov_default'], static_size=cs['size']))
    RO = pd.DataFrame(rows); save(RO, 'M4a_rolling_origin'); print(RO.round(3).to_string(index=False))

with section('4b. random-split controls (is the shortfall temporal?)'):
    rows = []; pk = PIPES['paper: F0 + LGBM default']; p = P[pk]
    for ctrl, yy in (('within 2012', 2012), ('within 2015', 2015)):
        idx = np.where(yr == yy)[0]; covs = []
        for sd in range(50):
            r_ = np.random.default_rng(sd); perm = r_.permutation(idx); h = len(perm) // 2; cal_i, ev_i = perm[:h], perm[h:]
            q = conf_q(true_scores(p[cal_i], y[cal_i]), ALPHA); S = np.column_stack([p[ev_i] <= q, (1 - p[ev_i]) <= q]); covs.append(S[np.arange(len(ev_i)), y[ev_i]].mean())
        rows.append(dict(control=f'calibrate on random half of {yy}, evaluate on the other half', mean_cov=np.mean(covs), sd=np.std(covs), min=np.min(covs), max=np.max(covs), repeats=50))
    S = conformal_sets(p, y, yr == 2012); m = yr == 2015; cs = cov_stats(S['static'][m], y[m])
    rows.append(dict(control='calibrate on ALL of 2012 -> evaluate 2015 (the temporal case)', mean_cov=cs['cov'], sd=np.nan, min=np.nan, max=np.nan, repeats=1))
    CTRL = pd.DataFrame(rows); save(CTRL, 'M4b_random_split_controls'); print(CTRL.round(4).to_string(index=False))

with section('4c. alpha sensitivity'):
    rows = []
    for pname, key in PIPES.items():
        for a in (0.05, 0.10, 0.20):
            S = conformal_sets(P[key], y, yr == 2012, alpha=a)
            for lab in ('static', 'class-conditional'):
                for yy in (2013, 2014, 2015):
                    m = yr == yy; cs = cov_stats(S[lab][m], y[m]); rows.append(dict(pipeline=pname, alpha=a, method=lab, year=yy, target=1 - a, cov=cs['cov'], cov_default=cs['cov_default'], size=cs['size'], share_reject_only=cs['share_reject_only']))
    ALPHAS = pd.DataFrame(rows); save(ALPHAS, 'M4c_alpha_sensitivity'); print(ALPHAS[ALPHAS.year == 2015].round(3).to_string(index=False))

In [ ]:
# ---------- 5. 60-month replication (train 2010-11, calibrate 2012, test 2013 by quarter) ----------
with section('5. 60-month replication'):
    assert W60 is not None, "60-month wide file missing"
    d = W60.reset_index(drop=True); y6, yr6, mo6 = d['default'].values, d['year'].values, d['month'].values; q6 = (mo6 - 1) // 3 + 1
    print("60-month loans:", len(d), "| by year:", d.groupby('year')['default'].agg(['size', 'mean']).round(3).to_dict('index'))
    rows = []; qrows = []
    for fs in ('F0 with int_rate', 'F1 no lender score', 'F3 applicant-only rich'):
        nums, cats = FS[fs]; X6 = design(d, nums, cats)
        for kind in ('LGBM default', 'LGBM regularised'):
            p = fit_predict(kind, X6, y6, yr6 <= 2011); S = conformal_sets(p, y6, yr6 == 2012)
            for lab, m in [('2012 (calibration)', yr6 == 2012)] + [(f'2013 Q{k}', (yr6 == 2013) & (q6 == k)) for k in (1, 2, 3, 4)] + [('2013 all', yr6 == 2013)]:
                cs = cov_stats(S['static'][m], y6[m]); cm = cov_stats(S['class-conditional'][m], y6[m])
                lo, hi = wilson(cs['cov'] * m.sum(), m.sum())
                rows.append(dict(feature_set=fs, model=kind, period=lab, N=int(m.sum()), actual_default=y6[m].mean(), mean_pred=p[m].mean(), AUC=roc_auc_score(y6[m], p[m]),
                                 static_cov=cs['cov'], ci_lo=lo, ci_hi=hi, static_cov_default=cs['cov_default'], static_size=cs['size'], classcond_cov=cm['cov'], classcond_cov_default=cm['cov_default'], classcond_size=cm['size']))
            idx = np.where(yr6 == 2012)[0]; cv = []
            for sd in range(50):
                perm = np.random.default_rng(sd).permutation(idx); h = len(perm) // 2; q = conf_q(true_scores(p[perm[:h]], y6[perm[:h]]), ALPHA)
                S_ = np.column_stack([p[perm[h:]] <= q, (1 - p[perm[h:]]) <= q]); cv.append(S_[np.arange(len(perm) - h), y6[perm[h:]]].mean())
            rows.append(dict(feature_set=fs, model=kind, period='iid control: random halves of 2012', N=len(idx) // 2, static_cov=np.mean(cv)))
    R60 = pd.DataFrame(rows); save(R60, 'M5_60month_replication')
    print(R60[R60.period.isin(['2013 all', 'iid control: random halves of 2012'])].round(3).to_string(index=False))

In [ ]:
# ---------- figures ----------
with section('figures'):
    # (a) decision gains vs approve-all, pooled 2013-2015
    if isinstance(DEC, pd.DataFrame) and len(DEC):
        pn = list(PIPES)[0]; D = DEC[(DEC.period == '2013-15') & (DEC.pipeline == pn) & (DEC.policy != 'Approve all') & (~DEC.policy.str.startswith('Oracle'))].copy()
        D = D[D.rule.isin(['single tier', 'approve-only (review->reject)', 'review->approve', 'expert, cost 0.0%', 'expert, cost 1.0%'])]
        D['label'] = D['policy'].str.replace(' (tuned on 2012 profit)', ' *tuned*', regex=False) + ' | ' + D['rule'].str.replace('approve-only (review->reject)', 'review=reject', regex=False)
        D = D.sort_values('gain_vs_approve_all'); fig, ax = plt.subplots(figsize=(11, 0.42 * len(D) + 1.5))
        ax.barh(D['label'], D['gain_vs_approve_all'], xerr=[D['gain_vs_approve_all'] - D['gain_ci_lo'], D['gain_ci_hi'] - D['gain_vs_approve_all']], color='steelblue', capsize=3)
        ax.axvline(0, c='k', lw=1); ax.set_xlabel('realised profit gain vs approving everyone, $ per applicant (95% month-block bootstrap)'); ax.set_title(f'Decision value on real outcomes, 2013-2015 ({pn})', fontweight='bold')
        plt.tight_layout(); savefig('M1_decision_gain'); plt.show()
    # (b) coverage by year per method (paper pipeline)
    pn = list(PIPES)[0]; R = DELAY_RES[pn]; R = R[R.period.isin(['2013', '2014', '2015'])]; fig, ax = plt.subplots(1, 2, figsize=(16, 5))
    for meth, g in R.groupby(['method', 'delay_months']):
        ax[0].plot(g['period'], g['cov'], marker='o', label=f'{meth[0]} (delay {meth[1]})'); ax[1].plot(g['period'], g['cov_default'], marker='o')
    ax[0].axhline(0.9, ls='--', c='k'); ax[0].set_title('Coverage by test year', fontweight='bold'); ax[0].legend(fontsize=7, loc='lower left'); ax[0].set_ylim(0.82, 0.96)
    ax[1].axhline(0.9, ls='--', c='k'); ax[1].set_title('Coverage of actual defaults', fontweight='bold'); plt.tight_layout(); savefig('M2_methods_under_delay'); plt.show()
    # (c) rolling origin
    fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
    for k, fs in enumerate(RO.feature_set.unique()):
        for Cy, g in RO[RO.feature_set == fs].groupby('calib_year'):
            ax[k].errorbar(g['lead'], g['static_cov'], yerr=[g['static_cov'] - g['ci_lo'], g['ci_hi'] - g['static_cov']], marker='o', capsize=3, label=f'calibrated {Cy}')
        ax[k].axhline(0.9, ls='--', c='k'); ax[k].set_title(f'{fs}: static coverage vs years after calibration', fontweight='bold'); ax[k].set_xlabel('years after calibration year'); ax[k].set_xticks([1, 2, 3, 4]); ax[k].legend()
    plt.tight_layout(); savefig('M3_rolling_origin'); plt.show()
    # (d) 60-month
    g6 = R60[R60.period.str.startswith('2013 Q')]; fig, ax = plt.subplots(figsize=(8, 4.5))
    for (fs, mdl), g in g6.groupby(['feature_set', 'model']): ax.plot(g['period'], g['static_cov'], marker='o', label=f'{fs}, {mdl}')
    ax.axhline(0.9, ls='--', c='k'); ax.set_title('60-month loans: static coverage by 2013 quarter', fontweight='bold'); ax.legend(fontsize=7); plt.tight_layout(); savefig('M4_60month'); plt.show()

In [ ]:
# ---------- summary, failures, zip ----------
lines = [f"MEGA ANALYSIS SUMMARY ({time.strftime('%Y-%m-%d %H:%M')}, runtime {(time.time() - T0) / 60:.1f} min)", f"Failed sections: {FAILED or 'none'}", ""]
def add(title, df_, cols=None, n=60):
    lines.append(f"## {title}"); lines.append(df_[cols].round(4).head(n).to_string(index=False) if cols is not None else df_.round(4).head(n).to_string(index=False)); lines.append("")
with section('summary'):
    add('Model zoo (val logloss on 2011 used for selection; test years not used)', ZOO, ['feature_set', 'model', 'val_logloss_2011', 'AUC_2013', 'AUC_2015', 'calib_gap_pp_2015', 'static_cov_2013', 'static_cov_2014', 'static_cov_2015', 'classcond_cov_default_2015'])
    for pn, r_ in DELAY_RES.items(): add(f'Methods under delay, 2015 and pooled: {pn}', r_[r_.period.isin(['2015', '2013-15'])])
    if isinstance(DEC, pd.DataFrame) and len(DEC): add('Decisions on realised profit, pooled 2013-15', DEC[DEC.period == '2013-15'], ['pipeline', 'policy', 'rule', 'share_approved', 'share_review', 'profit_per_applicant', 'gain_vs_approve_all', 'gain_ci_lo', 'gain_ci_hi', 'roi_on_approved'])
    add('Rolling origin', RO); add('Random-split controls', CTRL); add('60-month replication', R60[R60.period.isin(['2013 all', 'iid control: random halves of 2012'])])
(OUT/'results_summary.txt').write_text("\n".join(lines))
print("\n".join(lines)[:6000])
print(f"\nFAILED SECTIONS: {FAILED or 'none'}")
zpath = shutil.make_archive(str(Q2/'outputs_mega'), 'zip', OUT); print("\nSend me this file:", zpath)